## A well engineered system prompt

In [1]:
# BAD system prompt -- too vague, no constraints
BAD_SYSTEM = "You are a helpful assistant. Answer the user's question."
# This will cause the LLM to hallucinate from training data
# when the answer is not clearly in the context.
# GOOD system prompt -- specific constraints and fallback behaviour
GOOD_SYSTEM = """You are a precise document assistant for KUST university.
Answer the user's question using ONLY the information provided in the
Context section below. Do not use any knowledge from your training data.
Rules:
- If the answer is clearly stated in the context, answer concisely and
cite the source document in parenth
eses, e.g. (Source: policy.pdf).
- If the answer is NOT in the context, respond with:
"I could not find this information in the provided documents."
- Do not infer, guess, or expand beyond what the context explicitly states.
- Keep answers concise and factual.
- If multiple sources provide relevant information, synthesise them and
cite all relevant sources."""
# CONTEXT INJECTION TEMPLATE
def build_rag_prompt(retrieved_chunks, user_question):
  context_block = "\n\n".join([
f"[Source: {chunk['source']}]\n{chunk['text']}"
for chunk in retrieved_chunks
])
  user_message = f"""Context:
{context_block}
  Question: {user_question}
  Answer:"""
  return GOOD_SYSTEM, user_message
# Example usage
chunks = [
{"source": "fee_structure.pdf", "text": "The semester fee is PKR 85,000."},
{"source": "fee_structure.pdf", "text": "Late payment incurs a 5% penalty."},
]
system, user = build_rag_prompt(chunks, "What happens if I pay my fees late?")
print("SYSTEM:\n", system[:100], "...")
print("USER:\n", user)

SYSTEM:
 You are a precise document assistant for KUST university.
Answer the user's question using ONLY the  ...
USER:
 Context:
[Source: fee_structure.pdf]
The semester fee is PKR 85,000.

[Source: fee_structure.pdf]
Late payment incurs a 5% penalty.
  Question: What happens if I pay my fees late?
  Answer:


## Structured Output (Json)

In [ ]:
# Structured output approach -- forces JSON response
STRUCTURED_SYSTEM = """You are a document QA assistant.
Answer using ONLY the provided context.
You MUST respond with valid JSON in this exact format:
{
"answer": "your concise answer here",
"sources": ["source1.pdf", "source2.pdf"],
"confidence": "high|medium|low",
"found_in_context": true
}
If the answer is not in the context, set found_in_context to false
and answer with "This information was not found in the documents."
Do not add any text outside the JSON object."""
# Parsing the structured response
import json
def parse_rag_response(llm_output_text):
 try:
# Strip any markdown code fences if present
  clean = llm_output_text.strip()
  if clean.startswith("```"):
    clean = clean.split("```")[1]
    if clean.startswith("json"):
       clean = clean[4:]
       
  return json.loads(clean)
 except json.JSONDecodeError:
  return {"answer": llm_output_text, "sources": [], "found_in_context": None}

## RAGAS Demo

In [1]:
# pip install ragas datasets langchain openai
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, \
context_precision, context_recall
from datasets import Dataset
# RAGAS requires a labelled evaluation dataset with 4 fields:
# question: the user query
# answer: the RAG system's generated answer
# contexts: list of retrieved chunks (as strings) used to generate the answer
# ground_truth: the correct reference answer (for recall computation)
eval_data = {
"question": [
"What documents are required for admission?",
"What is the semester fee and when is it due?",
],
"answer": [
"Students must submit CNIC and academic transcripts. (Source: admission_policy.pdf)",
"The semester fee is PKR 85,000, payable in two installments. (Source: fee_structure.pdf)",
],
"contexts": [
["Students must submit CNIC and transcripts for admission to KUST."],
["The semester fee is PKR 85,000 payable in two installments.",
"Late payment incurs a 5% penalty after the due date."],
],
"ground_truth": [
"CNIC copy and academic transcripts are required for admission.",
"The semester fee is PKR 85,000 due in two installments.",
]
}
dataset = Dataset.from_dict(eval_data)
# Run evaluation -- requires an OpenAI API key for LLM-based metrics
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ["OPENAI_API_KEY"] = "your-key-here"
result = evaluate(
dataset,
metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)
print(result)
# Output example:
# {
#"faithfulness": 0.97,-- almost all claims grounded in context
#"answer_relevancy": 0.91,-- answers are relevant to questions
#"context_precision": 0.85,-- most retrieved chunks were useful
#"context_recall": 0.88-- found most of what was needed
# }
# You can also access per-question scores
df = result.to_pandas()
print(df[["question", "faithfulness", "answer_relevancy"]])

/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_24527/767625250.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, \
/tmp/ipykernel_24527/767625250.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, \
/tmp/ipykernel_24527/767625250.py:3: DeprecationWarning: Importing co

{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': nan}


KeyError: "['question'] not in index"

In [1]:
# pip install langchain langchain-community langchain-openai chromadb
# pip install sentence-transformers openai

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

# ── Step 1: Prepare your documents ───────────────────────────
raw_docs = [
    Document(
        page_content="Students must submit CNIC and transcripts for admission.",
        metadata={"source": "admission_policy.pdf"}
    ),
    Document(
        page_content="The semester fee is PKR 85,000 payable in two installments.",
        metadata={"source": "fee_structure.pdf"}
    ),
    Document(
        page_content="Late payment incurs a 5% penalty after the due date.",
        metadata={"source": "fee_structure.pdf"}
    ),
    Document(
        page_content="Full refund available for withdrawal within first two weeks.",
        metadata={"source": "fee_structure.pdf"}
    ),
    Document(
        page_content="The library is open from 8am to 10pm on weekdays.",
        metadata={"source": "hostel_policy.pdf"}
    ),
]

# ── Step 2: Chunk (if needed -- already small in this example) ─
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(raw_docs)

print(f"Total chunks: {len(chunks)}")

# ── Step 3: Embed and store in ChromaDB ──────────────────────
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./rag_db"
)

print(f"Stored {vectorstore._collection.count()} chunks in ChromaDB")

# ── Step 4: Create retriever ──────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
    # retrieve top-3 chunks
)

# ── Step 5: Set up LLM ────────────────────────────────────────
# Using OpenAI -- replace with any supported LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.0,
    # deterministic, factual
    max_tokens=500
)

# ── Step 6: Build RetrievalQA chain ──────────────────────────
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # "stuff" = put all retrieved docs in one prompt
    retriever=retriever,
    return_source_documents=True,   # also return which chunks were used
    verbose=True
)

# ── Step 7: Query ─────────────────────────────────────────────
query = "What is the penalty for paying fees late?"
result = qa_chain({"query": query})

print("\nAnswer:")
print(result["result"])

print("\nSources used:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}: {doc.page_content[:60]}...")

/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 5


/tmp/ipykernel_30581/669056993.py:42: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13949.86it/s]


Stored 5 chunks in ChromaDB


/tmp/ipykernel_30581/669056993.py:83: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": query})




> Entering new RetrievalQA chain...


RateLimitError: Error code: 429 - {'error': {'message': 'Your account is not active, please check your billing details on our website.', 'type': 'billing_not_active', 'param': None, 'code': 'billing_not_active'}}